# Step 4 - Virality model + factor analysis

Predict the virality level (low / medium / high) of a post from pre-post content/context features,
then rank the factors that drive virality.
All features are known *before* publishing (no view/like/comment counts) to avoid label leakage.

In [5]:
# load features + pick target
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(ROOT / "ml" / "data" / "video_features.parquet")

num_features = [
    "title_len_words", "desc_len_words", "trans_len_words",
    "title_sentiment", "title_has_question", "title_upper_ratio",
    "kw_price", "kw_range", "kw_charging",
    "pub_hour", "pub_dow", "pub_month",
    "duration_min", "channel_freq", "has_description", "has_transcript",
]
X = df[num_features]
y = df["virality_class"]                 # 0=low, 1=medium, 2=high
print("X:", X.shape, "| classes:", y.value_counts().to_dict())

X: (515, 16) | classes: {2: 172, 0: 172, 1: 171}


In [6]:
# baseline + cross-validated evaluation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# reference baseline: always predict the majority class
dummy = DummyClassifier(strategy="most_frequent")
dummy_pred = cross_val_predict(dummy, X, y, cv=cv)
print("Baseline macro-F1:", round(f1_score(y, dummy_pred, average="macro"), 3))

rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=3,
                            random_state=42, n_jobs=-1)
pred = cross_val_predict(rf, X, y, cv=cv)      # honest out-of-fold predictions
print("RandomForest macro-F1:", round(f1_score(y, pred, average="macro"), 3))
print("\n", classification_report(y, pred, target_names=["low", "medium", "high"]))
print("Confusion (rows=true):\n", confusion_matrix(y, pred))

Baseline macro-F1: 0.261
RandomForest macro-F1: 0.549

               precision    recall  f1-score   support

         low       0.62      0.67      0.64       172
      medium       0.47      0.36      0.41       171
        high       0.56      0.65      0.60       172

    accuracy                           0.56       515
   macro avg       0.55      0.56      0.55       515
weighted avg       0.55      0.56      0.55       515

Confusion (rows=true):
 [[115  28  29]
 [ 50  61  60]
 [ 20  41 111]]


In [7]:
# factor importance via permutation importance on a held-out set
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
rf.fit(Xtr, ytr)

perm = permutation_importance(rf, Xte, yte, n_repeats=20,
                              random_state=42, scoring="f1_macro")
imp = (pd.DataFrame({"feature": num_features,
                     "importance": perm.importances_mean,
                     "std": perm.importances_std})
       .sort_values("importance", ascending=False))
print("=== Factors driving VIRALITY (high to low) ===")
print(imp.to_string(index=False))

=== Factors driving VIRALITY (high to low) ===
           feature  importance      std
   trans_len_words    0.168991 0.031840
      duration_min    0.075597 0.021438
    desc_len_words    0.058266 0.014709
   title_len_words    0.035690 0.023228
           pub_dow    0.023479 0.011252
      channel_freq    0.021275 0.015442
          pub_hour    0.021032 0.007080
   has_description    0.015708 0.007796
   title_sentiment    0.015608 0.012781
         pub_month    0.013896 0.011778
          kw_range    0.012583 0.011150
 title_upper_ratio    0.011566 0.022204
          kw_price    0.011294 0.011824
       kw_charging    0.009721 0.008839
    has_transcript    0.006274 0.009852
title_has_question   -0.005036 0.008326


In [8]:
# readable single-factor analysis (direction of effect)
look = df.copy()
look["is_high"] = (look["virality_class"] == 2).astype(int)   # top-tier virality
for col in ["title_has_question", "kw_price", "kw_range", "kw_charging", "has_transcript"]:
    print(f"\n% high-virality by {col}:")
    print(look.groupby(col)["is_high"].mean().round(3))


% high-virality by title_has_question:
title_has_question
0    0.339
1    0.321
Name: is_high, dtype: float64

% high-virality by kw_price:
kw_price
0    0.257
1    0.473
Name: is_high, dtype: float64

% high-virality by kw_range:
kw_range
0    0.326
1    0.347
Name: is_high, dtype: float64

% high-virality by kw_charging:
kw_charging
0    0.279
1    0.407
Name: is_high, dtype: float64

% high-virality by has_transcript:
has_transcript
0    0.071
1    0.375
Name: is_high, dtype: float64
